# Fig S5H — TF motif accessibility vs RNA expression across pseudotime
For each Fig 5D TF with a single gene (TFGroup_N motif groups excluded), a scatter of its TF motif accessibility coefficient (vs NE root) against its mean mRNA level, one point per pseudotime segment (colored NE root → differentiated tip). Spearman ρ per TF quantifies chromatin–transcriptome concordance. Concordant TFs (e.g. GRHL2, ρ≈+0.9) track together; shared-motif TFs (e.g. ZEB1) diverge because motif activity reflects a whole E-box family, not the single gene's expression.

In [ ]:
from paperfig_style import *
import numpy as np, pandas as pd, matplotlib.pyplot as plt


In [ ]:
from scipy.stats import spearmanr
coef = load_matrix('PseudotimeTFActivity_coef.csv')
seg = sorted([c for c in coef.columns if c.startswith('PT_seg')], key=lambda c: int(c.replace('PT_seg','')))
coef = coef[seg]
tr = pd.read_csv(os.path.join(CSV_DIR, 'PseudotimeTFActivity_trend.csv')); tr = tr[tr['nSigSeg'] >= 1]
rise = [g for g in tr.sort_values('trend', ascending=False)['TF'] if g in coef.index][:10]
fall = [g for g in tr.sort_values('trend')['TF'] if g in coef.index][:10]
mrna = load_matrix('mRNA_by_pseudotime_segment.csv'); mrna = mrna[[c for c in seg if c in mrna.columns]]
mrna = mrna[~mrna.index.duplicated(keep='first')]
# individual TFs only (motif groups discarded); alias the few HOCOMOCO names -> gene symbols
ALIAS = {'ZN317':'ZNF317','ZBT14':'ZBTB14','NDF1':'NEUROD1','NGN1':'NEUROG1','TWST1':'TWIST1',
         'GCR':'NR3C1','ANDR':'AR','PRGR':'PGR','NF2L2':'NFE2L2','THA':'THRA','THB':'THRB'}
def to_gene(t):
    if t.startswith('TFGroup_'): return None       # discard motif groups
    if t in mrna.index: return t
    return ALIAS.get(t) if ALIAS.get(t) in mrna.index else None
MAIN = {'ZEB1','GRHL2','NFIC','ZBT14'}   # shown in main Fig 5E; the rest go here
pairs = [(t, to_gene(t)) for t in rise + fall if t not in MAIN]
tfs = [(t, g) for t, g in pairs if g is not None]
dropped = [t for t, g in pairs if g is None and not t.startswith('TFGroup_')]
print('individual TFs plotted:', len(tfs), '|', [g for _, g in tfs])
print('dropped (absent from RNA reference):', dropped)
segidx = np.arange(1, len(seg)+1); cmap = segment_cmap(len(seg))
ncol = 3; nrow = int(np.ceil(len(tfs)/ncol))
fig, axs = plt.subplots(nrow, ncol, figsize=(2.2*ncol, 2.1*nrow), layout='constrained')
axs = np.atleast_1d(axs).ravel()
for k, (t, g) in enumerate(tfs):
    ax = axs[k]; x = coef.loc[t].values.astype(float); y = mrna.loc[g].values.astype(float)
    rho = spearmanr(x, y)[0]
    sc = ax.scatter(x, y, c=segidx, cmap=cmap, vmin=0.5, vmax=len(seg)+0.5, s=20, edgecolor='k', linewidths=0.3)
    d = 'rise' if t in rise else 'fall'
    ax.set_title(f'{g}  ({d}, \u03c1={rho:.2f})', fontsize=7)
    ax.axvline(0, color='grey', lw=0.4, ls='--')
    ax.set_xlabel('TF motif accessibility (coef vs root)', fontsize=6)
    ax.set_ylabel('mean mRNA', fontsize=6); ax.tick_params(labelsize=5)
for k in range(len(tfs), len(axs)): axs[k].axis('off')
cb = fig.colorbar(sc, ax=axs.tolist(), fraction=0.015, pad=0.02); cb.set_label('pseudotime segment', fontsize=6)
fig.suptitle('TF motif accessibility vs RNA expression across pseudotime', fontsize=8)
savepanel(fig, 'FigS5H_TFActivity_vs_mRNA')
